# GeoSentinel-AI Semantic Backbone Training (Phase 4)
This notebook trains the DeepLabV3+ model on the pre-generated `.npy` patches to achieve 150-epoch "Elite" semantic convergence.

### Pre-requisites for Kaggle:
1. You must zip your local `data/benchmark/real/train` and `data/benchmark/real/val` folders into a file called `geosentinel_semantic_data.zip`.
2. Upload that zip file to Kaggle as a **New Dataset** (Add Input -> Upload Data).

In [ ]:
!pip install torch torchvision torchgeo lightning segmentation-models-pytorch rasterio pystac-client planetary-computer

In [ ]:
import os

# Clone the repository if it doesn't exist
if not os.path.exists('GeoSentinel-AI'):
    !git clone https://github.com/karthikeya-bhamidipati/GeoSentinel-AI.git

os.chdir('GeoSentinel-AI')

In [ ]:
import os
import shutil
from pathlib import Path

# Auto-detect the Kaggle dataset path
def find_train_dir(start_path="/kaggle/input"):
    for root, dirs, files in os.walk(start_path):
        if "train" in dirs and "val" in dirs:
            return root
    return None

dataset_path = find_train_dir()

if dataset_path:
    print(f"Found dataset at: {dataset_path}")
    data_dir = Path("data/benchmark/real")
    data_dir.mkdir(parents=True, exist_ok=True)
    
    if os.path.exists(data_dir / "train"):
        shutil.rmtree(data_dir / "train")
    if os.path.exists(data_dir / "val"):
        shutil.rmtree(data_dir / "val")
        
    print("Symlinking data folders...")
    os.symlink(os.path.join(dataset_path, "train"), data_dir / "train")
    os.symlink(os.path.join(dataset_path, "val"), data_dir / "val")
    print("Symlink successful! Ready for training.")
else:
    print("ERROR: Could not find 'train' and 'val' folders in /kaggle/input!")
    print("Did you definitely upload the zip file via 'Add Input'?")

In [ ]:
# Launch 150-epoch Semantic Segmentation Training
!python scripts/train.py --model deeplabv3plus --epochs 150 --batch-size 8

In [ ]:
import shutil
from IPython.display import FileLink

# Move the best weights to root for easy downloading
if os.path.exists('data/weights/deeplabv3plus_best.pt'):
    shutil.copy('data/weights/deeplabv3plus_best.pt', '/kaggle/working/deeplabv3plus_elite.pt')
    display(FileLink(r'/kaggle/working/deeplabv3plus_elite.pt'))
else:
    print("Training did not complete or weights were not saved.")